# Ontology-Constrained Memory (OCM) — Google Colab runner

A write-time **governed** memory layer for long-horizon LLM agents. This notebook runs the project end to end:

1. Get the code & install deps
2. Sanity tests
3. **Offline governance demo** (no GPU, no API key)
4. Benchmark + metrics (baselines B0–B3)
5. Full experiment suite — offline reference (multi-seed CIs, significance, τ-sweep, stress)
6. *(optional)* Real embeddings (`sentence-transformers`)
7. *(optional)* Local **Qwen2.5-32B-Instruct** extractor + **7b. the full research run** (Qwen + real embeddings)

Sections 1–6 run on a **CPU-only** runtime. Section 7 needs an **A100 80GB** GPU runtime (Runtime → Change runtime type → GPU).

## 1. Get the code

**Option A — clone from GitHub** (public repo). **Option B** (next cell) — upload a zip or mount Google Drive.

In [ ]:
# Option A: clone from a Git remote.
import os, getpass
REPO_URL = "https://github.com/tysjosh/ocmr.git"  # public; for a private repo, supply a token below
REPO_DIR = "/content/ocmr"
TOKEN = getpass.getpass("GitHub token (press Enter for the public repo): ").strip()
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL
if not os.path.exists(REPO_DIR):
    rc = os.system(f"git clone {url} {REPO_DIR}")
    if rc != 0:
        print("Clone failed — use Option B (upload/Drive) below.")
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    print("cwd:", os.getcwd())

In [ ]:
# Option B (only if you did NOT clone): upload a zip of the project, or mount Drive.
# from google.colab import files
# up = files.upload()                      # choose ocmr.zip
# !unzip -q ocmr.zip -d /content && ls /content
# %cd /content/ocmr                        # adjust if the zip nests a folder
#
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/ocmr          # adjust path

## 2. Install dependencies

The core (offline) demo needs only a few light packages. `chromadb` and `sentence-transformers` are **optional** — OCM falls back to a pure-Python vector index and a deterministic embedding provider when they are absent.

In [ ]:
# Light core install (enough for sections 2–5).
!pip -q install "pydantic>=2.6,<3" "networkx>=3.2" "fastapi>=0.110" "uvicorn>=0.29" "httpx>=0.27" "pytest>=8.0" "hypothesis>=6.100"

import sys, os
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import ocm
print("OCM importable from", ROOT)

## 2b. Persistent output (Google Drive)

Mount Drive so results **and per-(method, seed) checkpoints** survive a Colab refresh/crash — a resumed run skips already-finished work instead of restarting. Set `USE_DRIVE = False` to keep outputs only on the (ephemeral) Colab disk.

In [ ]:
import os
USE_DRIVE = True
OUTPUT_DIR = "/content/ocm_results"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_DIR = "/content/drive/MyDrive/ocm_results"
    except Exception as e:
        print("Drive mount failed; using local dir:", e)
CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)
print("Outputs ->", OUTPUT_DIR)
print("Checkpoints ->", CKPT_DIR)

## 3. Sanity tests

In [ ]:
!python -m pytest -q ocm/tests/test_has_status_assertions.py ocm/tests/test_experiment_stats.py ocm/tests/test_transformers_extractor.py

## 4. Offline governance demo (no GPU, no API key)

Facts are accepted, a status flip is **quarantined** (not silently overwritten), a correction **supersedes**, and the status query surfaces the contradiction inline.

In [ ]:
from ocm.core.config import Settings
from ocm.core.container import CoreContainer

c = CoreContainer(Settings(deterministic_test_mode=True, chroma_mode="memory", extractor="mock"))

def show(r, label):
    print(f"\n# {label}")
    print("  accepted   :", [o.candidate.predicate for o in r.accepted])
    print("  superseded :", [o.candidate.predicate for o in r.superseded])
    print("  quarantined:", [o.reason for o in r.quarantined])

show(c.write_pipeline.run("Alice owns Project Orion. Bob is assigned to Task T1.", "s1"), "W1 ownership + assignment")
show(c.write_pipeline.run("Bob completed Task T1.", "s2"), "W2 completion -> T1 done")
show(c.write_pipeline.run("Task T1 is not started.", "s3"), "W3 status flip -> QUARANTINED")
show(c.write_pipeline.run("Actually, Carol is assigned to Task T1.", "s4"), "W4 correction -> SUPERSEDE")

pkg = c.retrieval_pipeline.query("What is the current status of Task T1?", top_k=10)
print("\nQuery: 'What is the current status of Task T1?'")
print("  answer   :", pkg.answer)
print("  conflicts:", [{"accepted": cf.accepted, "quarantined": cf.quarantined, "reason": cf.reason} for cf in pkg.conflicts])

## 5a. Benchmark + metrics (baselines B0–B3)

In [ ]:
!python -m ocm.scripts.report_metrics --seed 1337

## 5b. Full experiment suite — offline reference run (multi-seed CIs, significance, τ-sweep, stress)

Runs the **full research protocol** (5 seeds × the full benchmark) on the offline mock extractor — a fast, deterministic reference. The genuine LLM-driven run is **Section 7b** (Qwen + real embeddings).

In [ ]:
!python -m ocm.scripts.run_experiments --seeds 1337 7 42 99 2024 \
  --checkpoint-dir {CKPT_DIR}/offline --out {OUTPUT_DIR}/results_offline.json

# Resumable: re-run after a crash/refresh and it skips finished (method, seed) work.
# This is the CPU-only reference run (offline mock extractor + deterministic
# embeddings). For the *real* LLM-driven research run (Qwen2.5-32B-Instruct +
# real embeddings) see Section 7b.
#
# Fast smoke run instead:  !python -m ocm.scripts.run_experiments --quick

## 6. (Optional) Real embeddings

Swaps the deterministic hashing embeddings for the real `all-MiniLM-L6-v2` model (downloads ~90 MB on first run). Runs on CPU or GPU.

In [ ]:
!pip -q install "sentence-transformers>=2.6"

from ocm.core.config import Settings
from ocm.core.container import CoreContainer

s = Settings(deterministic_test_mode=False, extractor="mock", embedding_mode="local",
             sqlite_path=":memory:", chroma_mode="memory")
c = CoreContainer(s)
c.write_pipeline.run("Alice owns Project Orion. Bob is assigned to Task T1.", "s1")
print("owner answer:", c.retrieval_pipeline.query("Who owns Project Orion?", top_k=5).answer)

## 7. (Optional) Local Qwen extractor — in-process via `transformers`

Loads **`Qwen/Qwen2.5-32B-Instruct`** in the same process (no server, no vLLM) and plugs it into OCM as the W1 extractor via `TransformersExtractor`.

Qwen2.5-32B-Instruct in bf16 is ~64 GB, so this needs an **A100 80GB** GPU runtime (Runtime → Change runtime type → GPU).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU — switch Runtime type to GPU."

In [ ]:
!pip -q install "transformers>=4.45" accelerate

In [ ]:
# Load Qwen2.5-32B-Instruct in-process (bf16, ~64 GB — needs an A100 80GB).
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("Loaded", MODEL_ID)

In [ ]:
# Plug the local model into OCM as the W1 extractor and run the governed pipeline.
from ocm.extraction.transformers_extractor import TransformersExtractor
from ocm.core.container import CoreContainer
from ocm.core.config import Settings

extractor = TransformersExtractor(model=llm_model, tokenizer=llm_tokenizer, max_new_tokens=1024)
# deterministic embeddings + in-memory storage keep everything else hermetic; the LLM does W1.
c = CoreContainer(Settings(deterministic_test_mode=True, chroma_mode="memory"), extractor=extractor)

for t, label in [
    ("Alice owns Project Orion. Bob is assigned to Task T1.", "W1"),
    ("Bob completed Task T1.", "W2"),
    ("Task T1 is not started.", "W3"),
]:
    r = c.write_pipeline.run(t, label)
    print(label, "| accepted", [o.candidate.predicate for o in r.accepted],
          "| quarantined", [bool(o.reason) for o in r.quarantined])

pkg = c.retrieval_pipeline.query("Who owns Project Orion?", top_k=5)
print("owner:", pkg.answer)
pkg = c.retrieval_pipeline.query("What is the current status of Task T1?", top_k=10)
print("status:", pkg.answer, "| conflicts:", [(cf.accepted, cf.quarantined) for cf in pkg.conflicts])

### 7b. Full research experiment — Qwen2.5-32B-Instruct + real embeddings

**This is the real research run.** It executes the entire protocol (baselines B0–B3, the four ablations, 5 seeds, the full benchmark, τ-sweep, stress) driven by the **local Qwen extractor** with **real `all-MiniLM-L6-v2` embeddings**. The model + embeddings are loaded **once** and shared across every arm.

Requires Section 6 (real embeddings installed) and Section 7 (Qwen loaded). **At full scale on a single A100 this can take many hours** — shrink `SEEDS` / `PER_CATEGORY` for a faster pass.

In [ ]:
import logging, os
from transformers import set_seed
from ocm.evaluation import experiment as exp
from ocm.extraction.transformers_extractor import TransformersExtractor
from ocm.retrieval.embeddings import LocalEmbeddingProvider

logging.getLogger("ocm").setLevel(logging.ERROR)  # quiet expected governance warnings

# Reproducibility: greedy decoding (do_sample=False) is already deterministic;
# this pins the global RNG too. The harness ALSO reseeds torch/transformers per
# seed inside run_full_suite, so every (method, seed) arm is reproducible.
set_seed(1337)

# Loaded once, shared across all baselines / ablations / seeds.
qwen_extractor = TransformersExtractor(model=llm_model, tokenizer=llm_tokenizer, max_new_tokens=1024)
# REAL all-MiniLM-L6-v2 embeddings (Section 6 installed it). NOTE: do NOT set
# deterministic_test_mode here — that would swap in fake hashing embeddings and
# disable the R2 semantic retriever. The full run uses real embeddings.
real_embeddings = LocalEmbeddingProvider()

SEEDS = [1337, 7, 42, 99, 2024]   # full protocol; reduce (e.g. [1337]) to shorten
PER_CATEGORY = 25                  # full benchmark; reduce (e.g. 5) to shorten

report = exp.run_full_suite(
    seeds=SEEDS,
    per_category=PER_CATEGORY,
    stress_per_class=30,
    extractor=qwen_extractor,
    embeddings=real_embeddings,
    checkpoint_dir=os.path.join(CKPT_DIR, "qwen"),          # resume on crash/refresh
    out_path=os.path.join(OUTPUT_DIR, "results_qwen.json"),  # final report on Drive
)
exp.print_report(report)
print("\nSaved ->", report.get("_saved_to"))

### Notes & caveats
- Sections 3–5 are fully offline/deterministic and need no GPU or keys.
- Section 6 (real embeddings) is the semantic retriever (R2) — part of the system, independent of the extractor; keep it for realistic retrieval.
- **Determinism:** the local extractor uses greedy decoding (`do_sample=False`), and the harness reseeds `torch`/`transformers`/`numpy`/`random` per seed inside `run_full_suite`, so each `(method, seed)` arm is reproducible. Report mean ± 95% CI across the 5 seeds (the harness computes CIs + Holm-Bonferroni significance) rather than relying on a single run.
- **Use the LLM extractor + real embeddings for the headline paper numbers** (Section 7b). Do **not** set `deterministic_test_mode=True` for the research run — it swaps in fake hashing embeddings. The mock extractor + `deterministic_test_mode` (Section 5b) is the offline *reference*, useful for CI/ablation sanity, not the main result.
- `Qwen/Qwen2.5-32B-Instruct` (bf16 ~64 GB) needs an **A100 80GB**.
- The in-process `TransformersExtractor` + real embeddings drive the **full research run (Section 7b)**: the model is loaded once and shared across every baseline/ablation/seed. Section 5b is the CPU-only offline reference.
- Full scale (5 seeds × full benchmark × 8 arms) with a 32B model can take **many hours** on one A100 — reduce `SEEDS` / `PER_CATEGORY` to shorten.
- **Crash-safe / resumable:** results and per-(method, seed) checkpoints are written to Drive (Section 2b). If Colab refreshes or crashes, just re-run the cell — finished work is skipped and the run resumes. Delete `OUTPUT_DIR/checkpoints` to force a clean rerun.
- OCM storage stays in-memory; the only files written are the results JSON and checkpoints under `OUTPUT_DIR`.
- Free the GPU when done: `del llm_model; import gc, torch; gc.collect(); torch.cuda.empty_cache()`.